<a href="https://colab.research.google.com/github/rsm-nshuckerow/Cribbage/blob/main/Cribbage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating DataFrame of all 5 card combos and points

In [ ]:
import pandas as pd
import itertools
import numpy as np
from google.colab import drive

## Rank, Suit, and Value List

In [ ]:
rank = ['A', 2, 3, 4, 5, 6, 7, 8, 9, 'T', 'J', 'Q', 'K']
suit = ['H', 'C', 'D', 'S']

In [ ]:
def get_card_value(card):
  if card in ['K', 'Q', 'J', 'T']:
    return 10
  elif card == 'A':
    return 1
  elif type(card) == int:
    return card


In [ ]:
cards = []
for i in rank:
  for j in suit:
    cards.append({"Rank": i, "Suit" :j, "Value": get_card_value(i)})


In [ ]:
cards_df = pd.DataFrame(cards)
cards_df.head()


,Rank,Suit,Value
0,A,H,1
1,A,C,1
2,A,D,1
3,A,S,1
4,2,H,2


## Creating all 4-card combos in deck

In [ ]:
card_combos_4 = list(itertools.combinations(cards, 4))

card_combos_4_df = pd.DataFrame(card_combos_4)

card_combos_4_df.head()

,0,1,2,3
0,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 'A', 'Suit': 'S', 'Value': 1}"
1,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 2, 'Suit': 'H', 'Value': 2}"
2,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 2, 'Suit': 'C', 'Value': 2}"
3,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 2, 'Suit': 'D', 'Value': 2}"
4,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 2, 'Suit': 'S', 'Value': 2}"


In [ ]:
# Extracting each cell's dictionary to create columns.

normalized_data = {}

for idx, col in enumerate(card_combos_4_df.columns):
    normalized_data[f'Rank_{idx}'] = card_combos_4_df[col].apply(lambda x: x['Rank'])
    normalized_data[f'Suit_{idx}'] = card_combos_4_df[col].apply(lambda x: x['Suit'])
    normalized_data[f'Value_{idx}'] = card_combos_4_df[col].apply(lambda x: x['Value'])

# Create the normalized DataFrame
normalized_df = pd.DataFrame(normalized_data)

len(normalized_df)

270725

## Four Card Hand Value Function

In [ ]:
def total_value_four_card(row):
  # initializing total variable
  total = 0

  # making lists for each 4 card combo for their rank, suit, and value
  rank = [row[f'Rank_{i}'] for i in range(4)]
  suit = [row[f'Suit_{i}'] for i in range(4)]
  value = [row[f'Value_{i}'] for i in range(4)]


  #-----------4 OF A KIND-----------
  # checking for 4 of a kind
  # stops after finding a 3 of a kind. Need to work this into making 15 as its returning total after this.
  for four_kind in itertools.combinations(rank, 4):
    if len(set(four_kind)) == 1:
      total +=  12
      return total
  # VERIFIED

  #-----------3 OF A KIND-----------
  used_in_three_kind = set()
  # checking for 3 of a kind
  for three_kind in itertools.combinations(rank, 3):
    if len(set(three_kind)) == 1 & len(used_in_three_kind) != 1 :
      total += 6
      used_in_three_kind.add(three_kind[0])

  # VERIFIED

  seen_pairs = set()
  # checking for pairs. does not return total in case there are multiple pairs
  if len(used_in_three_kind) == 0:
    for pair in itertools.combinations(range(4), 2):
      i, j = pair
      if rank[i] == rank[j]:
          if (rank[i], suit[i], suit[j]) not in seen_pairs:
            total += 2
            seen_pairs.add((rank[i], suit[i], suit[j]))


  # VERIFIED

  rank_values = []

  for r in rank:
    if r == 'A':
      rank_values.append(1)
    elif r == 'T':
      rank_values.append(10)
    elif r == 'J':
      rank_values.append(11)
    elif r == 'Q':
      rank_values.append(12)
    elif r == 'K':
      rank_values.append(13)
    else:
      rank_values.append(r)

  sorted_rank = sorted(rank_values)
  four_to_run = 0
  if sorted_rank[1] == sorted_rank[0]+1 and \
     sorted_rank[2] == sorted_rank[1]+1 and \
     sorted_rank[3] == sorted_rank[2]+1:
    total += 4
    four_to_run = 1

  if four_to_run == 0:
    for three_to_run in itertools.combinations(sorted_rank, 3):
      if sorted(three_to_run)[0] + 1 == sorted(three_to_run)[1] and \
        sorted(three_to_run)[1] + 1 == sorted(three_to_run)[2]:
        total += 3

  # VERIFIED

  for four_to_15 in itertools.combinations(value, 4):
    if sum(four_to_15) == 15:
      total += 2
  # VERIFIED
  for three_to_15 in itertools.combinations(value, 3):
    if sum(three_to_15) == 15:
      total += 2

  for two_to_15 in itertools.combinations(value, 2):
    if sum(two_to_15) == 15:
      total += 2

  if len(set(suit)) == 1:
    total += 4

  return total


total_value_four_card(normalized_df.iloc[236062])

12

In [ ]:
normalized_df['4_Hand_Value'] = normalized_df.apply(total_value_four_card, axis=1)

In [ ]:
normalized_df.head()

,Rank_0,Suit_0,Value_0,Rank_1,Suit_1,Value_1,Rank_2,Suit_2,Value_2,Rank_3,Suit_3,Value_3,4_Hand_Value
0,A,H,1,A,C,1,A,D,1,A,S,1,12
1,A,H,1,A,C,1,A,D,1,2,H,2,6
2,A,H,1,A,C,1,A,D,1,2,C,2,6
3,A,H,1,A,C,1,A,D,1,2,D,2,6
4,A,H,1,A,C,1,A,D,1,2,S,2,6


## Adding 5th card to combos

In [ ]:


# Split the DataFrame into batches
batch_size = 1000
batches = np.array_split(normalized_df, len(normalized_df) // batch_size)

# Process each batch separately
five_card_combos = []
for batch in batches:
    for _, row in batch.iterrows():
        # Extract the existing 4-card hand
        existing_hand = [
            {"Rank": row[f"Rank_{i}"], "Suit": row[f"Suit_{i}"], "Value": row[f"Value_{i}"]}
            for i in range(4)
        ]

        # Generate possible fifth cards (not in the existing hand)
        remaining_cards = [card for card in cards if card not in existing_hand]

        # Add each remaining card as the fifth card
        for fifth_card in remaining_cards:
            new_hand = existing_hand + [fifth_card]  # Append the fifth card to the hand
            five_card_combos.append(new_hand)


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
five_card_combos_df = pd.DataFrame(five_card_combos)

In [ ]:
five_card_combos_df.head()

,0,1,2,3,4
0,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 'A', 'Suit': 'S', 'Value': 1}","{'Rank': 2, 'Suit': 'H', 'Value': 2}"
1,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 'A', 'Suit': 'S', 'Value': 1}","{'Rank': 2, 'Suit': 'C', 'Value': 2}"
2,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 'A', 'Suit': 'S', 'Value': 1}","{'Rank': 2, 'Suit': 'D', 'Value': 2}"
3,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 'A', 'Suit': 'S', 'Value': 1}","{'Rank': 2, 'Suit': 'S', 'Value': 2}"
4,"{'Rank': 'A', 'Suit': 'H', 'Value': 1}","{'Rank': 'A', 'Suit': 'C', 'Value': 1}","{'Rank': 'A', 'Suit': 'D', 'Value': 1}","{'Rank': 'A', 'Suit': 'S', 'Value': 1}","{'Rank': 3, 'Suit': 'H', 'Value': 3}"


In [ ]:
normalized_5_data = {}

for idx, col in enumerate(five_card_combos_df.columns):
    normalized_5_data[f'Rank_{idx}'] = five_card_combos_df[col].apply(lambda x: x['Rank'])
    normalized_5_data[f'Suit_{idx}'] = five_card_combos_df[col].apply(lambda x: x['Suit'])
    normalized_5_data[f'Value_{idx}'] = five_card_combos_df[col].apply(lambda x: x['Value'])

# Create the normalized DataFrame
normalized_5_df = pd.DataFrame(normalized_5_data)

normalized_5_df.head()


,Rank_0,Suit_0,Value_0,Rank_1,Suit_1,Value_1,Rank_2,Suit_2,Value_2,Rank_3,Suit_3,Value_3,Rank_4,Suit_4,Value_4
0,A,H,1,A,C,1,A,D,1,A,S,1,2,H,2
1,A,H,1,A,C,1,A,D,1,A,S,1,2,C,2
2,A,H,1,A,C,1,A,D,1,A,S,1,2,D,2
3,A,H,1,A,C,1,A,D,1,A,S,1,2,S,2
4,A,H,1,A,C,1,A,D,1,A,S,1,3,H,3


In [ ]:
len(normalized_5_df)

12994800

## Five Card Hand Value Function

In [ ]:
def total_value_five_card(row, show_output=False):
  # initializing total variable
  total = 0

  # making lists for each 4 card combo for their rank, suit, and value
  rank = [row[f'Rank_{i}'] for i in range(5)]
  suit = [row[f'Suit_{i}'] for i in range(5)]
  value = [row[f'Value_{i}'] for i in range(5)]

  # checking for 4 of a kind
  # stops after finding a 3 of a kind. Need to work this into making 15 as its returning total after this.
  four_of_kind = 0
  for four_kind in itertools.combinations(rank, 4):
    if four_of_kind == 0 and len(set(four_kind)) == 1:
      total +=  12
      four_of_kind = 1
      if show_output == True:
        print(f"Points from Four of a Kind: {total}")

  # VERIFIED

  used_in_three_kind = set()
  three_of_kind = 0
  # checking for 3 of a kind
  if four_of_kind == 0:
    if three_of_kind == 0:
      for three_kind in itertools.combinations(rank, 3):
        if len(set(three_kind)) == 1:
          total += 6
          three_of_kind = 1
          used_in_three_kind.add(three_kind[0])
          if show_output == True:
            print(f"Points from Three of a Kind: {total}")

  # VERIFIED

  seen_pairs = set()
  # checking for pairs. does not return total in case there are multiple pairs
  for pair in itertools.combinations(range(5), 2):
    i, j = pair
    if rank[i] == rank[j] and rank[i] not in used_in_three_kind and four_of_kind == 0:
        if (rank[i], suit[i], suit[j]) not in seen_pairs:
          total += 2
          seen_pairs.add((rank[i], suit[i], suit[j]))
          if show_output == True:
            print(f"Points from Pair: {total}")


  # VERIFIED

  rank_values = []

  for r in rank:
    if r == 'A':
      rank_values.append(1)
    elif r == 'T':
      rank_values.append(10)
    elif r == 'J':
      rank_values.append(11)
    elif r == 'Q':
      rank_values.append(12)
    elif r == 'K':
      rank_values.append(13)
    else:
      rank_values.append(r)

  sorted_rank = sorted(rank_values)
  five_to_run = 0
  if sorted_rank[1] == sorted_rank[0]+1 and \
     sorted_rank[2] == sorted_rank[1]+1 and \
     sorted_rank[3] == sorted_rank[2]+1 and \
     sorted_rank[4] == sorted_rank[3]+1:
     total += 5
     five_to_run = 1
     if show_output == True:
       print(f"Points from Run of Five: {total}")

  four_to_run = 0
  if five_to_run == 0:
    for four_run in itertools.combinations(sorted_rank, 4):
      if sorted(four_run)[1] == sorted(four_run)[0]+1 and \
        sorted(four_run)[2] == sorted(four_run)[1]+1 and \
        sorted(four_run)[3] == sorted(four_run)[2]+1:
        total += 4
        four_to_run = 1
        if show_output == True:
          print(f"Points from Run of Four: {total}")

  if four_to_run == 0 and five_to_run == 0:
    for three_to_run in itertools.combinations(sorted_rank, 3):
      if sorted(three_to_run)[0] + 1 == sorted(three_to_run)[1] and \
        sorted(three_to_run)[1] + 1 == sorted(three_to_run)[2]:
        total += 3
        if show_output == True:
          print(f"Points from Run of Three: {total}")

  # VERIFIED

  five_makes_15 = 0
  if sum(value) == 15:
      total += 2
      five_makes_15 = 1
      if show_output == True:
        print(f"Points from five cards to make 15: {total}")
  else:
    for four_to_15 in itertools.combinations(value, 4):
      if sum(four_to_15) == 15:
        total += 2
        if show_output == True:
          print(f"Points from four cards to make 15: {total}")
    # VERIFIED
    for three_to_15 in itertools.combinations(value, 3):
      if sum(three_to_15) == 15:
        total += 2
        if show_output == True:
          print(f"Points from three cards to make 15: {total}")

    for two_to_15 in itertools.combinations(value, 2):
      if sum(two_to_15) == 15:
        total += 2
        if show_output == True:
          print(f"Points from two cards to make 15: {total}")

  five_flush = 0
  if len(set(suit)) == 1:
    total += 5
    five_flush = 1
    if show_output == True:
      print(f"Points from 5-card flush: {total}")

  if len(set(suit[0:4])) == 1 & five_flush == 0:
    total += 4
    if show_output == True:
      print(f"Points from 4-card flush: {total}")

  # if J in rank and J suit is same as 5th card
  if 'J' in rank and suit[rank.index('J')] == suit[4]:
    total += 1
    if show_output == True:
      print(f"Points from Jack in a suit: {total}")

  if rank[4] == 'J':
    total += 2
    if show_output == True:
      print(f"Points from Jack on flop: {total}")

  return total

total_value_five_card(normalized_5_df.iloc[10168528])

29

In [ ]:
list(itertools.combinations([1,2,3,4,4], 4))

[(1, 2, 3, 4), (1, 2, 3, 4), (1, 2, 4, 4), (1, 3, 4, 4), (2, 3, 4, 4)]

In [ ]:
normalized_5_df['Total_4_card'] = normalized_5_df.apply(total_value_four_card, axis=1)
normalized_5_df['Total_5_card'] = normalized_5_df.apply(total_value_five_card, axis=1)
normalized_5_df.head()

,Rank_0,Suit_0,Value_0,Rank_1,Suit_1,Value_1,Rank_2,Suit_2,Value_2,Rank_3,Suit_3,Value_3,Rank_4,Suit_4,Value_4,Total_4_card,Total_5_card
0,A,H,1,A,C,1,A,D,1,A,S,1,2,H,2,12,12
1,A,H,1,A,C,1,A,D,1,A,S,1,2,C,2,12,12
2,A,H,1,A,C,1,A,D,1,A,S,1,2,D,2,12,12
3,A,H,1,A,C,1,A,D,1,A,S,1,2,S,2,12,12
4,A,H,1,A,C,1,A,D,1,A,S,1,3,H,3,12,12


# Mapping Hand Input to Hand DataFrame

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Save normalized_5_df to my google drive
normalized_5_df.to_csv('/content/drive/My Drive/Cribbage_Points_Matrix.csv', index=False)

In [ ]:
normalized_5_df = pd.read_csv('/content/drive/My Drive/Cribbage_Points_Matrix.csv')

/tmp/ipython-input-1066574123.py:1: DtypeWarning: Columns (0,3,6) have mixed types. Specify dtype option on import or set low_memory=False.
  normalized_5_df = pd.read_csv('/content/drive/My Drive/Cribbage_Points_Matrix.csv')


## Casting Rank Columns as string

Need to fix this in original dataframe code



In [ ]:
import numpy as np

for i in range(5):
  normalized_5_df[f"Rank_{i}"] = normalized_5_df[f"Rank_{i}"].astype(str)

In [ ]:
ranks = ['A', '2', '3', '4', '5', '6', '7', '8', '9', 'T', 'J', 'Q', 'K']
suits = ['H', 'C', 'D', 'S']
rank_values = {r:i for i,r in enumerate(ranks)}
suit_values = {s:i for i,s in enumerate(suits)}

rank_input_to_symbol = {
    'A':'A', 1:'A',
    'K':'K', 13:'K',
    'Q':'Q', 12:'Q',
    'J':'J', 11:'J',
    'T':'T', '10':'T', 10:'T',
    '2':'2', 2:'2','3':'3',3:'3','4':'4',4:'4','5':'5',5:'5',
    '6':'6',6:'6','7':'7',7:'7','8':'8',8:'8','9':'9',9:'9'
}

## Mapping function from rank/suit to order value

In [ ]:
import pandas as pd
import numpy as np



def _rank_series_to_codes(s: pd.Series) -> np.ndarray:
    return s.map(rank_values).to_numpy(np.int16)


def _suit_series_to_codes(s: pd.Series) -> np.ndarray:
    return s.map(suit_values).to_numpy(np.int16)

## Bitmasking four and five card hands

In [ ]:
def make_bitwise_card_id(df:pd.DataFrame) -> np.ndarray:
    mask = np.zeros(len(df), dtype=np.uint64)
    for i in range(4):
        rank_codes = _rank_series_to_codes(df[f'Rank_{i}'])
        suit_codes = _suit_series_to_codes(df[f'Suit_{i}'])
        card_id = (13*suit_codes + rank_codes).astype(np.int16)
        mask |= (np.uint64(1) << card_id.astype(np.uint64))
    return mask

In [ ]:
four_card_index = make_bitwise_card_id(normalized_5_df)

In [ ]:
def make_bitwise_card_id_v2(df:pd.DataFrame) -> np.ndarray:
    mask = np.zeros(len(df), dtype=np.uint64)
    for i in range(4,5):
        rank_codes = _rank_series_to_codes(df[f'Rank_{i}'])
        suit_codes = _suit_series_to_codes(df[f'Suit_{i}'])
        card_id = (13*suit_codes + rank_codes).astype(np.int16)
        mask |= (np.uint64(1) << card_id.astype(np.uint64))
    return mask

In [ ]:
fifth_card_index = make_bitwise_card_id_v2(normalized_5_df)

In [ ]:
normalized_5_df = normalized_5_df.set_index([four_card_index, fifth_card_index], drop=True)

## Calculating bitmask from Hand

In [ ]:
import pandas as pd
test_card = ['2C', '3C', '4C', '5C']

def calculate_4_hand_index(hand):
  mask = np.uint64(0)
  for card in hand:
    card_rank = card[0]
    card_suit = card[1]
    rank_code = _rank_series_to_codes(pd.Series([card_rank]))
    suit_code = _suit_series_to_codes(pd.Series([card_suit]))
    card_id = (13*suit_code + rank_code).astype(np.int16)
    mask |= (np.uint64(1) << card_id.astype(np.uint64))
  return mask[0]

def calculate_fifth_card_index(card):
  mask = np.uint64(0)
  card_rank = card[0]
  card_suit = card[1]
  rank_code = _rank_series_to_codes(pd.Series([card_rank]))
  suit_code = _suit_series_to_codes(pd.Series([card_suit]))
  card_id = (13*suit_code + rank_code).astype(np.int16)
  mask |= (np.uint64(1) << card_id.astype(np.uint64))
  return mask[0]

In [ ]:
index_value_1 = calculate_4_hand_index(test_card)
index_value_2 = calculate_fifth_card_index('9C')

index_value_1, index_value_2



(np.uint64(245760), np.uint64(2097152))

In [ ]:
import itertools

test_cards = ['2C', '3C', '4C', '5C', '6C', '7C']

def four_card_combos_from_six(hand):
  combos = list(itertools.combinations(hand, 4))



In [ ]:
combos = list(itertools.combinations(test_cards, 4))
bitmasks = [calculate_4_hand_index(i) for i in combos]
fifth_card_bitmask = [calculate_fifth_card_index(i) for i in test_cards]

In [ ]:
# Create a boolean mask by checking if the first level of the index (4-card bitmask)
# is in the list of bitmasks.
bitmask_set = set(bitmasks)
fifth_card_bitmask_set = set(fifth_card_bitmask)

mask = normalized_5_df.index.get_level_values(0).map(bitmask_set.__contains__)
fifth_mask = normalized_5_df.index.get_level_values(1).map(fifth_card_bitmask_set.__contains__)

# Apply the mask to filter the DataFrame
filtered_df = normalized_5_df[mask & ~fifth_mask]

# Display the filtered DataFrame
display(filtered_df)

Rank_0 Suit_0  Value_0 Rank_1 Suit_1  Value_1 Rank_2  \
245760 1                     2      C        2      3      C        3      4   
       8192                  2      C        2      3      C        3      4   
       67108864              2      C        2      3      C        3      4   
       549755813888          2      C        2      3      C        3      4   
       2                     2      C        2      3      C        3      4   
...                        ...    ...      ...    ...    ...      ...    ...   
983040 1125899906842624      4      C        4      5      C        5      6   
       4096                  4      C        4      5      C        5      6   
       33554432              4      C        4      5      C        5      6   
       274877906944          4      C        4      5      C        5      6   
       2251799813685248      4      C        4      5      C        5      6   

                        Suit_2  Value_2 Rank_3 Suit_3  Value_3 Rank_4 Suit_4  \
245760 1                     C        4      5      C        5      A      H   
       8192                  C        4      5      C        5      A      C   
       67108864              C        4      5      C        5      A      D   
       549755813888          C        4      5      C        5      A      S   
       2                     C        4      5      C        5      2      H   
...                        ...      ...    ...    ...      ...    ...    ...   
983040 1125899906842624      C        6      7      C        7      Q      S   
       4096                  C        6      7      C        7      K      H   
       33554432              C        6      7      C        7      K      C   
       274877906944          C        6      7      C        7      K      D   
       2251799813685248      C        6      7      C        7      K      S   

                         Value_4  Total_4_card  Total_5_card  
245760 1                       1             8             7  
       8192                    1             8            12  
       67108864                1             8             7  
       549755813888            1             8             7  
       2                       2             8            10  
...                          ...           ...           ...  
983040 1125899906842624       10            10             8  
       4096                   10            10             8  
       33554432               10            10            13  
       274877906944           10            10             8  
       2251799813685248       10            10             8  

[690 rows x 17 columns]

## Highest Guaranteed Points

In [ ]:
s = filtered_df.groupby(level=0)['Total_4_card'].agg('mean')

s_max = s.max()

winners = s[s == s_max].index

normalized_5_df.loc[winners]

Rank_0 Suit_0  Value_0 Rank_1 Suit_1  Value_1 Rank_2  \
491520 1                     3      C        3      4      C        4      5   
       8192                  3      C        3      4      C        4      5   
       67108864              3      C        3      4      C        4      5   
       549755813888          3      C        3      4      C        4      5   
       2                     3      C        3      4      C        4      5   
...                        ...    ...      ...    ...    ...      ...    ...   
983040 1125899906842624      4      C        4      5      C        5      6   
       4096                  4      C        4      5      C        5      6   
       33554432              4      C        4      5      C        5      6   
       274877906944          4      C        4      5      C        5      6   
       2251799813685248      4      C        4      5      C        5      6   

                        Suit_2  Value_2 Rank_3 Suit_3  Value_3 Rank_4 Suit_4  \
491520 1                     C        5      6      C        6      A      H   
       8192                  C        5      6      C        6      A      C   
       67108864              C        5      6      C        6      A      D   
       549755813888          C        5      6      C        6      A      S   
       2                     C        5      6      C        6      2      H   
...                        ...      ...    ...    ...      ...    ...    ...   
983040 1125899906842624      C        6      7      C        7      Q      S   
       4096                  C        6      7      C        7      K      H   
       33554432              C        6      7      C        7      K      C   
       274877906944          C        6      7      C        7      K      D   
       2251799813685248      C        6      7      C        7      K      S   

                         Value_4  Total_4_card  Total_5_card  
491520 1                       1            10             8  
       8192                    1            10            13  
       67108864                1            10             8  
       549755813888            1            10             8  
       2                       2            10             9  
...                          ...           ...           ...  
983040 1125899906842624       10            10             8  
       4096                   10            10             8  
       33554432               10            10            13  
       274877906944           10            10             8  
       2251799813685248       10            10             8  

[96 rows x 17 columns]

## Highest Possible Points

In [ ]:
filtered_df['Total_5_card'].max()

NameError: name 'filtered_df' is not defined

## Highest Estimated Mean Value

In [ ]:
s = filtered_df.groupby(level=0)['Total_5_card'].agg('mean')

s_max = s.max()

winners = s[s == s_max].index

normalized_5_df.loc[winners]

Rank_0 Suit_0  Value_0 Rank_1 Suit_1  Value_1 Rank_2  \
491520 1                     3      C        3      4      C        4      5   
       8192                  3      C        3      4      C        4      5   
       67108864              3      C        3      4      C        4      5   
       549755813888          3      C        3      4      C        4      5   
       2                     3      C        3      4      C        4      5   
       16384                 3      C        3      4      C        4      5   
       134217728             3      C        3      4      C        4      5   
       1099511627776         3      C        3      4      C        4      5   
       4                     3      C        3      4      C        4      5   
       268435456             3      C        3      4      C        4      5   
       2199023255552         3      C        3      4      C        4      5   
       8                     3      C        3      4      C        4      5   
       536870912             3      C        3      4      C        4      5   
       4398046511104         3      C        3      4      C        4      5   
       16                    3      C        3      4      C        4      5   
       1073741824            3      C        3      4      C        4      5   
       8796093022208         3      C        3      4      C        4      5   
       32                    3      C        3      4      C        4      5   
       2147483648            3      C        3      4      C        4      5   
       17592186044416        3      C        3      4      C        4      5   
       64                    3      C        3      4      C        4      5   
       524288                3      C        3      4      C        4      5   
       4294967296            3      C        3      4      C        4      5   
       35184372088832        3      C        3      4      C        4      5   
       128                   3      C        3      4      C        4      5   
       1048576               3      C        3      4      C        4      5   
       8589934592            3      C        3      4      C        4      5   
       70368744177664        3      C        3      4      C        4      5   
       256                   3      C        3      4      C        4      5   
       2097152               3      C        3      4      C        4      5   
       17179869184           3      C        3      4      C        4      5   
       140737488355328       3      C        3      4      C        4      5   
       512                   3      C        3      4      C        4      5   
       4194304               3      C        3      4      C        4      5   
       34359738368           3      C        3      4      C        4      5   
       281474976710656       3      C        3      4      C        4      5   
       1024                  3      C        3      4      C        4      5   
       8388608               3      C        3      4      C        4      5   
       68719476736           3      C        3      4      C        4      5   
       562949953421312       3      C        3      4      C        4      5   
       2048                  3      C        3      4      C        4      5   
       16777216              3      C        3      4      C        4      5   
       137438953472          3      C        3      4      C        4      5   
       1125899906842624      3      C        3      4      C        4      5   
       4096                  3      C        3      4      C        4      5   
       33554432              3      C        3      4      C        4      5   
       274877906944          3      C        3      4      C        4      5   
       2251799813685248      3      C        3      4      C        4      5   

                        Suit_2  Value_2 Rank_3 Suit_3  Value_3 Rank_4 Suit_4  \
491520 1               